# Phase 4 (Part B): Labour Analysis

This notebook queries the `labour_attendance` table and performs analytical checks:
1. Explores labour attendance trends by site.
2. Calculates the correlation between a project's average attendance rate (%) and its timeline delay (days) to verify the impact of attendance on timeline performance.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append src folder to path
sys.path.append(os.path.abspath('../src'))

import db_connection

## 1. Load Data from SQLite

In [ ]:
conn = db_connection.get_connection()
df_labour = pd.read_sql_query("SELECT * FROM labour_attendance", conn)
df_projects = pd.read_sql_query("SELECT * FROM projects", conn)
conn.close()

df_labour["date"] = pd.to_datetime(df_labour["date"])
df_projects["start_date"] = pd.to_datetime(df_projects["start_date"])
df_projects["planned_end_date"] = pd.to_datetime(df_projects["planned_end_date"])
df_projects["actual_end_date"] = pd.to_datetime(df_projects["actual_end_date"])

df_labour.head()

## 2. Calculate Project Delay Days

We use the same logic as Phase 3 to establish project delays.

In [ ]:
ref_date = pd.to_datetime("2026-07-18")

def calculate_delay(row):
    if pd.notna(row["actual_end_date"]):
        return (row["actual_end_date"] - row["planned_end_date"]).days
    elif row["status"] == "Delayed" or (row["status"] == "Ongoing" and ref_date > row["planned_end_date"]):
        return (ref_date - row["planned_end_date"]).days
    else:
        return 0

df_projects["delay_days"] = df_projects.apply(calculate_delay, axis=1)
df_projects_mini = df_projects[["project_id", "project_type", "status", "delay_days"]]
df_projects_mini.head()

## 3. Site Attendance Summary & Trends

In [ ]:
site_summary = df_labour.groupby("site_name").agg(
    avg_attendance_pct=("attendance_pct", "mean"),
    total_expected_mandays=("workers_expected", "sum"),
    total_present_mandays=("workers_present", "sum")
).reset_index()

site_summary["avg_attendance_pct"] = site_summary["avg_attendance_pct"].round(2)
site_summary

## 4. Correlation: Attendance Rate vs. Project Delay

Group labour attendance by project, compute average attendance rate, join with project delay data, and calculate Pearson correlation coefficient.

In [ ]:
proj_attendance = df_labour.groupby("project_id")["attendance_pct"].mean().reset_index(name="avg_proj_attendance")
df_corr = pd.merge(proj_attendance, df_projects_mini, on="project_id")

correlation = df_corr["avg_proj_attendance"].corr(df_corr["delay_days"])
print(f"Pearson Correlation between Project Attendance % and Project Delay (Days): {correlation:.4f}")

df_corr.head()

## 5. Visualizations

In [ ]:
sns.set_theme(style="whitegrid")

# Plot 1: Attendance Rate by Site (Boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_labour, x="site_name", y="attendance_pct", palette="Set3")
plt.title("Daily Attendance Rate Distribution by Site", fontsize=14, fontweight='bold')
plt.xlabel("Site Name", fontsize=12)
plt.ylabel("Attendance Rate (%)", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/site_attendance_distribution.png", dpi=300)
plt.show()

# Plot 2: Scatter Plot with regression line showing correlation
plt.figure(figsize=(10, 6))
sns.regplot(data=df_corr, x="avg_proj_attendance", y="delay_days", scatter_kws={'s':80}, line_kws={'color':'red'})
plt.title(f"Correlation: Average Attendance % vs. Project Delay (Pearson r = {correlation:.2f})", fontsize=14, fontweight='bold')
plt.xlabel("Average Project Attendance Rate (%)", fontsize=12)
plt.ylabel("Project Delay (Days)", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/attendance_vs_delay_correlation.png", dpi=300)
plt.show()